### A re-implementation of `GSEA_Mitochondria_FullDEGList.ipynb` for this project's environment / conventions

In [7]:
# setup: imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

from utils.paths import DATA, FIGURES
from matplotlib.figure import Figure
from pathlib import Path

from adjustText import adjust_text

In [8]:
# initialize values & filepaths

# volcano
THRESH_padj = 0.5
THRESH_lfc = 0.5
TOP_N_GENES = 25

# filepaths
DEG_FILEPATHS = [fp for fp in DATA.glob("deseq.results.*.txt")]
SEP = r"\s+"

# gsea
PRESELECTED_TERMS = {
    # "Oxidative Phosphorylation",
    # "Xenobiotic Metabolism",
    # "Fatty Acid Metabolism",
    # "Reactive Oxygen Species Pathway",
    # "Glycolysis",
    # "heme Metabolism",
}
TOP_N_TERMS = 5  # per sample, 6 in total figure
TOP_N_TERMS_METRIC = "NES"  # normalized enrichment score

In [9]:
# utility functions


def preprocess_deseq_df(deseq_df: pd.DataFrame) -> pd.DataFrame:
    """
    preprocess deseq df in-place
    - remove dirty rows (non-kgCluster ids)
    - convert numeric columns from string to numeric dtypes
    """

    deseq_df = deseq_df[
        deseq_df["id"].str.startswith("kgCluster")
    ]  # filter bad columns
    deseq_df["symbol"] = (
        deseq_df["symbol"].astype(str).str.upper()
    )  # uppercase gene names

    base_numeric_cols = ["log2FcMAP", "lfcSE", "stat", "pvalue", "padj"]
    custom_numeric_cols = [
        col for col in deseq_df.columns if col.startswith("baseMean")
    ]

    for col in base_numeric_cols + custom_numeric_cols:
        deseq_df[col] = pd.to_numeric(deseq_df[col], errors="coerce")

    return deseq_df


def get_ranked_list(deseq_df: pd.DataFrame) -> pd.DataFrame:
    return (
        df[["symbol", "log2FcMAP"]]
        .set_index("symbol")
        .sort_values(by="log2FcMAP", ascending=False)
    )


def preprocess_volcano(
    deseq_df: pd.DataFrame,
    gene_set: set | pd.Index,
    padj_thresh: float,
    lfc_thresh: float,
) -> pd.DataFrame:
    """
    Returns a preprocessed slice of `deseq_df` with a significance column ready to plot in a volcano plot.

    :param deseq_df: The dataframe. Expects gene IDs to be dataframe index, not a column.
    :type deseq_df: pd.DataFrame
    :param gene_set: Set of genes to contribute to the volcano plot (usually those associated with a GO term)
    :type gene_set: set | pd.Index
    :param padj_thresh: The adjusted p-value threshold. Operates on column 'padj'. Strictly lower values are significant
    :type padj_thresh: float
    :param lfc_thresh: The log fold change threshold. Operates on column 'log2FcMAP'. Strictly lower values than -lfc_thresh are significant (downreg) and strictly greater
    """

    deseq_df = deseq_df[deseq_df["symbol"].isin(gene_set)]

    padj_sig = deseq_df["padj"] < padj_thresh
    upreg = deseq_df["log2FcMAP"] > lfc_thresh
    downreg = deseq_df["log2FcMAP"] < -lfc_thresh

    deseq_df["significance"] = "Not sig."
    deseq_df.loc[padj_sig & upreg, "significance"] = "Up"
    deseq_df.loc[padj_sig & downreg, "significance"] = "Down"

    return deseq_df


def plot_volcano(
    deseq_df: pd.DataFrame,
    padj_thresh: float,
    lfc_thresh: float,
    symbol: str,
    top_n_genes: int | None,
    figsize: tuple[int, int] = (8, 6),
    alpha: float = 0.7,
    dot_size: int = 10,
    save: Path | None = None,
    show: bool = True,
) -> Figure:

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

    x, y = deseq_df["log2FcMAP"], -np.log10(deseq_df["padj"])
    cmap = deseq_df["significance"].map(
        {"Up": "red", "Down": "blue", "Not sig.": "grey"}
    )

    ax.scatter(x, y, c=cmap, alpha=alpha, s=dot_size)

    ax.axhline(-np.log10(padj_thresh), color="black", linestyle="--", lw=1)
    ax.axvline(x=lfc_thresh, color="black", linestyle="--", lw=1)
    ax.axvline(x=-lfc_thresh, color="black", linestyle="--", lw=1)

    ax.set_xlabel("log2(Fold Change)")
    ax.set_ylabel("-log10(adjusted p-value)")
    fig.suptitle(f"Symbol: {symbol}")

    if top_n_genes:
        top_n_entries = deseq_df.sort_values(by="padj").head(top_n_genes)
        annots = [
            ax.text(
                x=row["log2FcMAP"],
                y=-np.log10(row["padj"]),
                s=row["symbol"],
                fontsize=8,
                ha="right" if row["log2FcMAP"] < 0 else "left",
            )
            for i, row in top_n_entries.iterrows()
        ]
        adjust_text(
            texts=annots,
            arrowprops=dict(arrowstyle="->", color="black"),
            force_text=(0.2, 0.4),
        )

    if save:
        fig.savefig(save)
    if show:
        fig.show()

    return fig


def pathway_specific_gsea(df: pd.DataFrame, terms: dict, outdir: Path) -> None:
    for go_term, gene_list in terms.items():
        filename = Path(go_term.replace(" ", "-")).with_suffix(".pdf")
        df.plot(go_term, ofname=str(outdir / filename))


def pathway_specific_volcano(
    df: pd.DataFrame,
    terms: dict[str, list[str]],
    outdir: Path | None,
    padj_thresh: float = 0.5,
    lfc_thresh: float = 0.5,
    top_n_genes: int = 20,
    show: bool = False,
):
    """
    Compute & generate volcano plots for every GO term in terms.
    If passed, plots are saved to the given output directory and
    titled based on the given GO term.

    :param df: deseq dataframe
    :type df: pd.DataFrame
    :param terms: dictionary mapping go terms to a list of genes
    :type terms: dict[str, list[str]]
    :param outdir: save volcano plots to this directory.
    :type outdir: Path
    :param padj_thresh: Adjusted p-value threshold. Genes >= than this are deemed not significant.
    :type padj_thresh: float
    :param lfc_thresh: Log2FoldChange threshold. Genes <= than this are deemed not significant.
    :type lfc_thresh: float
    :param top_n_genes: N most significant genes to label based on adjusted p-value.
    :type top_n_genes: int
    :param show: whether to render the generated figure.
    :type show: bool
    """
    outdir.mkdir(parents=True, exist_ok=True)

    for go_term, gene_list in terms.items():
        filename = Path(go_term.replace(" ", "-")).with_suffix(".png")
        outfp = outdir / filename if outdir else None

        volcano_df = preprocess_volcano(
            deseq_df=df,
            gene_set=set(gene_list),
            padj_thresh=padj_thresh,
            lfc_thresh=lfc_thresh,
        )
        fig = plot_volcano(
            deseq_df=volcano_df,
            padj_thresh=padj_thresh,
            lfc_thresh=lfc_thresh,
            symbol=go_term,
            top_n_genes=top_n_genes,
            save=outfp,
            show=False,
        )
        plt.close(fig)


def get_top_n_terms(prerank_obj: gp.Prerank, n_per_sample: int = 3, metric: str = "NES"):
    """
    Get the top n terms (by sample) in a gsea dataframe by the given metric. Scoring is 
    done using metric absolute values. Pathway names are prepended with their positional
    ranking (e.g. "Top Pathway" -> "[1] Top Pathway"), with `prerank_obj.results` receiving
    a duplicate entry with the altered name as to not break `prerank_obj.plot`.

    :param df: (gseapy.prerank object).res2d
    :type df: pd.DataFrame
    :param n: number of entries to include
    :type n_per_sample: int
    :param metric: metric in top n with respect to metric, column name in prerankobj.res2d
    :type metric: str
    """
    df = prerank_obj.res2d.copy().sort_values(metric, ascending=False)
    df = pd.concat((df[:n_per_sample], df[-n_per_sample:]), axis=0, ignore_index=True)

    abs_metric = "abs" + metric
    df[abs_metric] = abs(df[metric])
    df.sort_values(abs_metric, ascending=False)

    df["TermOld"] = df["Term"]
    df["Term"] = "[" + (df.index + 1).astype(str) + "] " + df["Term"]

    # needed as prerank_obj.plot() compares prerank_obj.res2d["Term"] and prerank_obj.results
    term_map = dict(zip(df["TermOld"], df["Term"]))
    for old, new in term_map.items():
        prerank_obj.results[new] = prerank_obj.results[old]

    return df


def filter_dict(d: dict, s: dict | set):
    if isinstance(s, dict):
        s = s.keys()
    return {k: d[k] for k in s}


def get_mitocarta_library():
    mitocarta_path = DATA/"Mouse.MitoCarta3.0.xls"
    if not mitocarta_path.exists():
        import subprocess
        subprocess.exec(["wget", "-P", str(DATA.resolve()), "https://personal.broadinstitute.org/scalvo/MitoCarta3.0/Mouse.MitoCarta3.0.xls"])

    df = pd.read_excel(mitocarta_path, sheet_name="C MitoPathways")
    ser = pd.Series(data=df['Genes'].tolist(), index=df['MitoPathway'])
    ser = ser.str.upper()
    return ser.str.split(", ").to_dict()



In [10]:
# get GO terms

mouse_libraries = gp.get_library_name(organism="Mouse")
lib_name = "MSigDB_Hallmark_2020"
assert lib_name in mouse_libraries
go_terms = gp.get_library(name=lib_name, organism="Mouse")

# new library: use mitocarta
# comment out if MSigDB_Hallmark_2020 is to be used
go_terms = get_mitocarta_library()

In [11]:
df = preprocess_deseq_df(deseq_df=pd.read_csv(DEG_FILEPATHS[0], sep=SEP))
ranked_list = get_ranked_list(deseq_df=df)
preranked_gsea = gp.prerank(
        rnk=ranked_list,
        gene_sets=go_terms,
        threads=8,
        min_size=5,
        max_size=1000,
        permutation_num=1000,
        outdir=None,
        seed=6,
        verbose=True,
    )

preranked_gsea.res2d

2026-03-14 23:20:57,373 [INFO] Input gene rankings contains duplicated IDs
2026-03-14 23:20:57,388 [WARNING] Duplicated values found in preranked stats: 5.58% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-03-14 23:20:57,389 [INFO] Parsing data files for GSEA.............................
2026-03-14 23:20:57,462 [INFO] 0036 gene_sets have been filtered out when max_size=1000 and min_size=5
2026-03-14 23:20:57,463 [INFO] 0113 gene_sets used for further statistical testing.....
2026-03-14 23:20:57,464 [INFO] Start to run GSEA...Might take a while..................
2026-03-14 23:21:00,263 [INFO] Congratulations. GSEApy runs successfully................



,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,OXPHOS subunits,0.726493,3.507006,0.0,0.0,0.0,61/82,18.23%,UQCRC1;NDUFA12;NDUFS7;COX4I1;NDUFB8;NDUFB9;NDU...
1,prerank,Mitochondrial ribosome,0.672487,3.288519,0.0,0.0,0.0,57/81,20.81%,MRPS18A;MRPS24;MRPL28;MRPL40;MRPS26;MRPL4;MRPL...
2,prerank,OXPHOS,0.629413,3.228664,0.0,0.0,0.0,92/136,21.78%,UQCRC1;NDUFA12;NDUFS7;COX4I1;NDUFB8;NDUFB9;NDU...
3,prerank,CI subunits,0.758642,3.161149,0.0,0.0,0.0,30/36,18.23%,NDUFA12;NDUFS7;NDUFB8;NDUFB9;NDUFA11;NDUFA9;ND...
4,prerank,Complex I,0.690094,2.993327,0.0,0.0,0.0,38/54,18.80%,NDUFA12;NDUFS7;NDUFB8;NDUFB9;NDUFA11;NDUFA9;ND...
...,...,...,...,...,...,...,...,...,...,...
108,prerank,Amino acid metabolism,-0.196009,-0.608141,0.976507,1.0,1.0,7/82,8.94%,CHDH;GLS;AMT;NAGS;SLC25A21;DHTKD1;DBT
109,prerank,EF hand proteins,-0.261567,-0.608065,0.935401,1.0,1.0,1/11,7.10%,SLC25A23
110,prerank,GABA metabolism,-0.268883,-0.517506,0.970803,1.0,1.0,3/6,32.13%,ALDH5A1;ABAT;ADHFE1
111,prerank,Phospholipid metabolism,-0.194839,-0.513564,0.98111,1.0,1.0,2/22,7.62%,SERAC1;GPAM


In [12]:
# create GSEA / volcano plots

for fp in DEG_FILEPATHS:
    df = preprocess_deseq_df(deseq_df=pd.read_csv(fp, sep=SEP))

    ranked_list = get_ranked_list(deseq_df=df)

    preranked_gsea = gp.prerank(
        rnk=ranked_list,
        gene_sets=go_terms,
        threads=8,
        min_size=5,
        max_size=1000,
        permutation_num=1000,
        outdir=None,
        seed=6,
        verbose=True,
    )

    # setup paths
    gsea_dir = FIGURES / fp.stem / "gsea-plots"
    gsea_dir.mkdir(parents=True, exist_ok=True)
    volcano_dir = FIGURES / fp.stem / "volcano"
    volcano_dir.mkdir(parents=True, exist_ok=True)

    all_terms = go_terms.keys()
    preselected_terms = filter_dict(go_terms, PRESELECTED_TERMS)
    top_n_terms_df = get_top_n_terms(
        prerank_obj=preranked_gsea, n_per_sample=TOP_N_TERMS, metric="NES"
    )

    # GSEA: preselected terms (joint)
    if PRESELECTED_TERMS:
        preranked_gsea.plot(
            terms=preselected_terms.keys(),
            figsize=(4, 5),
            ofname=str(gsea_dir / "preselected-terms.pdf"),
        )

    # GSEA: top n terms
    preranked_gsea.plot(
        terms=top_n_terms_df["Term"].tolist(),
        figsize=(4, 3 + TOP_N_TERMS),
        ofname=str(gsea_dir / f"top-{TOP_N_TERMS}-terms-{TOP_N_TERMS_METRIC}.pdf"),
    )

    # GSEA: write preranked_gsea.res2d as figure metadata
    preranked_gsea.res2d.to_csv(gsea_dir / "gsea-metadata.csv", sep=',', index=False)

    # GSEA & Volcano plots: individual preselected terms
    # pathway_specific_gsea(df=preranked_gsea, terms=preselected_terms, outdir=gsea_dir)
    # pathway_specific_volcano(
    #     df=df,
    #     terms=preselected_terms,
    #     outdir=volcano_dir,
    #     padj_thresh=THRESH_padj,
    #     lfc_thresh=THRESH_lfc,
    #     top_n_genes=TOP_N_GENES,
    #     show=False,
    # )
    None

2026-03-14 23:21:00,539 [INFO] Input gene rankings contains duplicated IDs
2026-03-14 23:21:00,558 [WARNING] Duplicated values found in preranked stats: 5.58% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-03-14 23:21:00,560 [INFO] Parsing data files for GSEA.............................
2026-03-14 23:21:00,649 [INFO] 0036 gene_sets have been filtered out when max_size=1000 and min_size=5
2026-03-14 23:21:00,650 [INFO] 0113 gene_sets used for further statistical testing.....
2026-03-14 23:21:00,651 [INFO] Start to run GSEA...Might take a while..................
2026-03-14 23:21:03,588 [INFO] Congratulations. GSEApy runs successfully................

2026-03-14 23:21:04,864 [INFO] Input gene rankings contains duplicated IDs
2026-03-14 23:21:04,883 [WARNING] Duplicated values found in preranked stats: 1.10% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-03-14 23:21:04,884 [INFO] Parsing data